In [ ]:
#build dataset_0 features + a group label (watchlist / undes_rest / designated)
features = pd.read_csv("../../data/final_datasets/dataset_0.csv", dtype={"plr_id": str})
features["plr_id"] = features["plr_id"].astype(str).str.zfill(8)

#attach y_true so we can separate designated from undesignated
features = features.merge(df[["plr_id", "y_true"]], on="plr_id", how="left")

#the urgency watchlist = top-25 undesignated PLR by urgency (oof_prob × unprotected share)
N_URGENTLIST = 25
urgentlist["urgency"] = urgentlist["oof_prob"] * (1 - urgentlist["milieu_anteil"])
watchlist_ids = set(
    urgentlist.sort_values("urgency", ascending=False)
              .head(N_URGENTLIST)["plr_id"]
)

def assign_group(row):
    if row["plr_id"] in watchlist_ids:
        return "watchlist"          # the urgency top-25
    if row["y_true"] == 1:
        return "designated"         # already protected
    return "undes_rest"             # undesignated but not on the watchlist

features["group"] = features.apply(assign_group, axis=1)
print(features["group"].value_counts().to_string())

In [ ]:
# --- vulnerable-population read: is there still a displaceable population? -----
# LEVEL = how much vulnerable pop is still present (the actual question)
# CHANGE = whether displacement already appears to be underway
# Descriptive only; watchlist selected on oof_prob, so read as characterization.


vuln_cols = [c for c in features.columns
             if any(k in c for k in ["transfer_benefit", "unemployment"])]
print("vulnerability-related columns found:", vuln_cols, "\n")

# group means for level + change side by side
prof = features.groupby("group")[vuln_cols].mean().T
prof = prof[["watchlist", "undes_rest", "designated"]].round(4)
print(prof.to_string(), "\n")

# per-PLR view of the watchlist: level AND change together, so you can see
# which areas still have a population AND whether it's already declining
level_col  = [c for c in vuln_cols if "transfer_benefit" in c and "change" not in c]
change_col = [c for c in vuln_cols if "transfer_benefit" in c and "change" in c]

if level_col and change_col:
    lv, ch = level_col[0], change_col[0]
    wl_view = (
        features[features["group"] == "watchlist"]
        [["plr_id", "plr_name", "bez", lv, ch]]     # names/bez direkt aus features
        .sort_values(lv, ascending=False)
    )
    print(f"Urgent list by transfer-benefit LEVEL ({lv}) with CHANGE ({ch}):")
    print(wl_view.to_string(index=False))

### The remaining vulnerable population

Across the 25 urgency-watchlist areas, the mean transfer-benefit share in 2024 is 0.113, sitting between the undesignated remainder (0.092) and the already-designated areas (0.123) — the flagged neighbourhoods still retain a vulnerable population close to the level of those already protected. That share is falling faster on the watchlist (−0.017 over 2020–2024) than in the undesignated remainder (−0.007), and nearly as fast as in designated areas (−0.020), consistent with displacement pressure already underway rather than merely anticipated.

The per-area picture is bimodal and should not be reduced to the mean. One group combines a large remaining vulnerable population with a steep decline — led by Mitte areas such as Schulstraße (level 0.242, change −0.026), Brunnenstraße (0.239, −0.034), Humboldthain Nordwest (0.222, −0.039) and Koloniestraße (0.212, −0.051) — marking them as areas where both the need and the urgency for protection are highest. A second group, including Arkonaplatz (0.033), Invalidenstraße (0.037) and Wilhelmstraße (0.047), resembles designated areas on the structural features the model uses but retains little benefit-dependent population, implying that on this income-based indicator the displacement of lower-income residents may already have largely occurred. Transfer-benefit share captures only one dimension of vulnerability and should be read as proxy, not proof.